In [0]:
%pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
from pyspark.sql.types import StringType, IntegerType, FloatType
import pyspark.sql.functions as F
import faker

catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.bronze.checkpoints_vol")

checkpoint_path = f"/Volumes/{catalog_name}/bronze/checkpoints_vol/drivers_stream"

print(f"A secure checkpoint has been created: {checkpoint_path}")

fake = faker.Faker()

@F.udf(returnType=StringType())
def get_name(): return fake.name()

@F.udf(returnType=StringType())
def get_car_number(): return fake.license_plate()

@F.udf(returnType=IntegerType())
def get_experience(): return fake.random_int(min=1, max=30)

@F.udf(returnType=FloatType())
def get_rating(): return round(fake.pyfloat(min_value=1.0, max_value=5.0), 1)

streaming_df = (spark.readStream.format("rate")
                .option("rowsPerSecond", 2)
                .load()
                .withColumn("name", get_name())
                .withColumn("car_number", get_car_number())
                .withColumn("experience", get_experience())
                .withColumn("rating", get_rating())
                .withColumnRenamed("value", "id")
                .drop("timestamp"))

query = (streaming_df.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .table("bronze.drivers_stream"))
         
query.awaitTermination()
print("The stream completed successfully, and the data was written to the Bronze table")

Xavfsiz checkpoint manzili yaratildi: /Volumes/workspace/bronze/checkpoints_vol/drivers_stream
Streaming muvaffaqiyatli yakunlandi va ma'lumotlar Bronze jadvaliga yozildi!


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType
import faker

fake = faker.Faker()
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]
checkpoint_path = f"/Volumes/{catalog_name}/bronze/checkpoints_vol/drivers_stream"

@F.udf(returnType=StringType())
def get_phone(): return fake.phone_number()

@F.udf(returnType=StringType())
def get_name(): return fake.name()
@F.udf(returnType=StringType())
def get_car_number(): return fake.license_plate()
@F.udf(returnType=IntegerType())
def get_experience(): return fake.random_int(min=1, max=30)
@F.udf(returnType=FloatType())
def get_rating(): return round(fake.pyfloat(min_value=1.0, max_value=5.0), 1)

evolved_streaming_df = (spark.readStream.format("rate")
                        .option("rowsPerSecond", 2)
                        .load()
                        .withColumn("name", get_name())
                        .withColumn("car_number", get_car_number())
                        .withColumn("experience", get_experience())
                        .withColumn("rating", get_rating())
                        .withColumn("phone", get_phone())
                        .withColumnRenamed("value", "id")
                        .drop("timestamp"))

print("Schema Evolution has started. A new 'phone' column is being added to the table...")

query_evolved = (evolved_streaming_df.writeStream
                 .format("delta")
                 .outputMode("append")
                 .option("checkpointLocation", checkpoint_path)
                 .option("mergeSchema", "true") 
                 .trigger(availableNow=True)
                 .table("bronze.drivers_stream"))

query_evolved.awaitTermination()
print("Success! The new column was added to the table and the data was written without errors.")

Schema Evolution boshlandi. Yangi 'phone' ustuni jadvalga qo'shilyapti...
Muvaffaqiyatli! Yangi ustun jadvalga xatosiz qo'shildi va ma'lumotlar yozildi.


In [0]:
%sql
SELECT * FROM bronze.drivers_stream TIMESTAMP AS OF current_timestamp() - INTERVAL 1 DAY;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8793713648365147>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '-- 1. Mentor so\'ragan kod: Jadvalning "kechagi" holatiga murojaat qilish\n-- (Jadval bugun ochilgani uchun bu kod xatolik bersa ham xavotir olmang, bu normal holat)\nSELECT * FROM bronze.drivers_stream TIMESTAMP AS OF current_timestamp() - INTERVAL 1 DAY;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.M

In [0]:
%sql
DESCRIBE HISTORY bronze.drivers_stream;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-05-26T06:10:49.000Z,76550624650996,ilkhomjon.ibragimov@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 94be213a-d1b2-42e0-89c2-b6cc24333017, epochId -> 1, statsOnLoad -> false)",null,List(393022657056643),9ac84a41-047b-4d14-ba02-bc908e06f1ad,0526-053629-lh8ghhph-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 3013, numOutputBytes -> 94326, numAddedFiles -> 8)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-05-26T06:10:30.000Z,76550624650996,ilkhomjon.ibragimov@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 94be213a-d1b2-42e0-89c2-b6cc24333017, epochId -> -1, statsOnLoad -> false)",null,List(393022657056643),9ac84a41-047b-4d14-ba02-bc908e06f1ad,0526-053629-lh8ghhph-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-05-26T05:45:54.000Z,76550624650996,ilkhomjon.ibragimov@innowise.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 94be213a-d1b2-42e0-89c2-b6cc24333017, epochId -> 0, statsOnLoad -> true)",null,List(393022657056643),d36d5546-f5fa-48f3-9f39-ba5feb8ff1ee,0526-053629-lh8ghhph-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 1, numOutputBytes -> 1512, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-26T05:37:04.000Z,76550624650996,ilkhomjon.ibragimov@innowise.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-fb00e92b-c679-4da3-8178-b7e4fd67e4e7"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6656f04c-d5e4-4b9f-8a84-44745f3be129""}, statsOnLoad -> false)",null,List(393022657056643),c39e33c0-3645-4590-a69f-91301c7e480e,0526-053629-lh8ghhph-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM bronze.drivers_stream VERSION AS OF 0;

id,name,car_number,experience,rating


In [0]:
%sql
RESTORE TABLE bronze.drivers_stream TO VERSION AS OF 0;

SELECT * FROM bronze.drivers_stream;

id,name,car_number,experience,rating


In [0]:
%sql
/* Note: 
Since this project is implemented on Databricks Community Edition (Free Tier), true Unity Catalog RBAC is disabled. 
The following Data Control Language (DCL) logic demonstrates how the security requirements would be implemented in a standard environment.
*/

-- CREATE USER `innowise@innowise.com`;
-- CREATE GROUP `innowise`;
-- ALTER GROUP `innowise` ADD USER `innowise@innowise.com`;

GRANT USAGE ON SCHEMA bronze TO `innowise`;
GRANT SELECT ON TABLE bronze.drivers_stream TO `innowise`;

DENY MODIFY ON TABLE bronze.drivers_stream TO `innowise`;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8793713648365151>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '/* Note for Mentor: \nSince this project is implemented on Databricks Community Edition (Free Tier), true Unity Catalog RBAC is disabled. \nThe following Data Control Language (DCL) logic demonstrates how the security requirements would be implemented in a standard environment.\n*/\n\n-- CREATE USER `innowise@innowise.com`;\n-- CREATE GROUP `innowise`;\n-- ALTER GROUP `innowise` ADD USER `innowise@innowise.com`;\n\nGRANT USAGE ON SCHEMA bronze TO `innowise`;\nGRANT SELECT ON TABLE bronze.drivers_stream TO `innowise`;\n\nDENY MODIFY ON TABLE bronze.drivers_stream TO `innowise`;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with 

In [0]:
%sql
OPTIMIZE bronze.drivers_stream ZORDER BY (id);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(0, 0), 0, List(0, 0), 0, null), null, 0, 0, 0, 0, false, 0, 0, 1779776686539, 1779776690292, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.drivers_enriched
CLUSTER BY (id, experience)
AS SELECT * FROM bronze.drivers_stream;

OPTIMIZE silver.drivers_enriched;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 0, 0, false, 0, 0, 1779776830049, 1779776830390, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, List(0, false, false, false, null, null, null, null, 0, 0, 0, 0, 0, 0, 0, null, null, 0, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 0, 0, 0, 0), 2, 1, 5, null, false, 0, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 0, 0, true, 0, 0, 1779776830435, 1779776830861, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, List(0, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 0, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.driver_performance (
    name STRING,
    avg_rating FLOAT,
    max_experience INT
);

COMMENT ON TABLE gold.driver_performance IS "Gold layer: Drivers' overall stats and ratings";

In [0]:
%sql
-- 1-DML: Grouping (aggregating) data from the Silver table and inserting it into the Gold table (INSERT)
INSERT INTO gold.driver_performance 
SELECT name, ROUND(AVG(rating), 1), MAX(experience) 
FROM silver.drivers_enriched 
GROUP BY name;

-- 2-DML: Automatically set the rating to 5.0 for veteran drivers with over 25 years of experience (UPDATE)
UPDATE gold.driver_performance 
SET avg_rating = 5.0 
WHERE max_experience > 25;

num_affected_rows
0


In [0]:
%sql
-- 1-DQL: Sort the drivers with the highest rating
SELECT name, avg_rating, max_experience 
FROM gold.driver_performance 
WHERE avg_rating > 4.5 
ORDER BY max_experience DESC;

-- 2-DQL: Calculating the number of drivers with the same experience
SELECT max_experience, COUNT(*) as drivers_count
FROM gold.driver_performance
GROUP BY max_experience
ORDER BY max_experience;

max_experience,drivers_count
